# 📊 Notebook 01 — Análisis Exploratorio de Datos (EDA)
**Dataset:** AI Student Impact Dataset  
**Objetivo:** Explorar la distribución, correlaciones y patrones del dataset para comprender la variable objetivo `Burnout_Risk_Level`.

In [ ]:
# ── Instalaciones (solo en Colab) ─────────────────────────────────────────
# !pip install seaborn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')
print('Librerías cargadas ✔')

In [ ]:
# ── Carga del dataset ─────────────────────────────────────────────────────
# En Colab: sube el archivo o usa drive.mount()
df = pd.read_csv('data/raw/ai_student_impact_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Información general ───────────────────────────────────────────────────
df.info()

In [ ]:
# ── Estadísticas descriptivas ─────────────────────────────────────────────
df.describe(include='all').T

In [ ]:
# ── Valores nulos ─────────────────────────────────────────────────────────
nulls = df.isnull().sum()
print('Valores nulos por columna:')
print(nulls[nulls >= 0].to_string())
print(f'\nTotal nulos: {nulls.sum()}')

In [ ]:
# ── Distribución de la variable objetivo ─────────────────────────────────
order = ['Low', 'Medium', 'High']
counts = df['Burnout_Risk_Level'].value_counts().reindex(order)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#4CAF50', '#FF9800', '#F44336']

axes[0].bar(order, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución de Burnout_Risk_Level', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad de estudiantes')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=11)

axes[1].pie(counts.values, labels=order, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción de clases', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('docs/figs/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nObservación: El dataset está moderadamente balanceado (Medium 42%, Low 33%, High 25%)')

In [ ]:
# ── Variables numéricas: histogramas ──────────────────────────────────────
num_cols = ['Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Tool_Diversity',
            'Traditional_Study_Hours', 'Perceived_AI_Dependency',
            'Anxiety_Level_During_Exams', 'Post_Semester_GPA', 'Skill_Retention_Score']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Frecuencia')

plt.suptitle('Distribución de Variables Numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('docs/figs/num_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Variables numéricas: boxplots vs. target ──────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
palette = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='Burnout_Risk_Level', y=col, order=order,
                palette=palette, ax=axes[i])
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Variables Numéricas vs. Burnout Risk Level', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('docs/figs/boxplots_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlación entre variables numéricas ────────────────────────────────
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlación — Variables Numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/figs/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nNota: Pre_Semester_GPA y Post_Semester_GPA muestran alta correlación positiva esperada.')

In [ ]:
# ── Variables categóricas ─────────────────────────────────────────────────
cat_cols = ['Major_Category', 'Year_of_Study', 'Primary_Use_Case',
            'Prompt_Engineering_Skill', 'Institutional_Policy']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['Burnout_Risk_Level'])[order]
    ct.plot(kind='bar', ax=axes[i], color=colors, edgecolor='white', linewidth=0.5)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(title='Burnout', loc='upper right', fontsize=8)

axes[5].axis('off')
plt.suptitle('Variables Categóricas vs. Burnout Risk Level', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/figs/categorical_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Pairplot de variables clave ───────────────────────────────────────────
key_cols = ['Weekly_GenAI_Hours', 'Traditional_Study_Hours',
            'Anxiety_Level_During_Exams', 'Skill_Retention_Score', 'Burnout_Risk_Level']

g = sns.pairplot(df[key_cols].sample(2000, random_state=42),
                 hue='Burnout_Risk_Level', hue_order=order,
                 palette=palette, plot_kws={'alpha': 0.4, 's': 20})
g.fig.suptitle('Pairplot — Variables Clave (muestra 2000 obs.)', y=1.02, fontsize=13)
plt.savefig('docs/figs/pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

## 📝 Conclusiones del EDA
- El dataset contiene **50,000 filas y 16 columnas** sin valores nulos.
- La variable objetivo `Burnout_Risk_Level` tiene tres clases (Low/Medium/High) con distribución moderadamente balanceada.
- Variables como `Anxiety_Level_During_Exams`, `Perceived_AI_Dependency` y `Weekly_GenAI_Hours` muestran diferencias notables entre clases → buenas features para KNN.
- Las variables numéricas tienen escalas muy distintas → **normalización obligatoria** para KNN.
- Variables categóricas requieren **Label Encoding** antes del modelado.